# AROGYA Fine-Tuning Experiments

This notebook demonstrates the Phase 6 fine-tuning workflow for the AROGYA medical assistant.
It acts as an interactive playground to verify the functionality of our dataset preparation, LoRA training, adapter merging, and evaluation scripts.

We will cover:
1. **Data Preparation**: Loading and formatting medical Q&A pairs for instruction tuning.
2. **LoRA Training**: Efficiently fine-tuning a base model (e.g., Llama-3-8B) using the PEFT library.
3. **Merging**: Fusing the trained adapter weights back into the base model.
4. **Evaluation**: Testing the fine-tuned model's capabilities on a hold-out test set.

In [ ]:
import os
import logging
from pathlib import Path

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Set up paths relative to the project root
PROJECT_ROOT = Path(os.getcwd()).parent
DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "models"

logger.info(f"Project root: {PROJECT_ROOT}")
logger.info(f"Data directory: {DATA_DIR}")
logger.info(f"Models directory: {MODEL_DIR}")

## 1. Data Preparation
Prepare the instruction-tuning dataset using `prepare_dataset.py`. This script converts raw JSONL files into Hugging Face Dataset objects, applies tokenization, and structures the prompts.

In [ ]:
from src.arogya.models.finetune.prepare_dataset import load_and_prepare_dataset

raw_data_path = DATA_DIR / "raw" / "medical_qa_sample.jsonl"
prepared_data_dir = DATA_DIR / "processed" / "finetune_dataset"

dataset = load_and_prepare_dataset(
    input_path=str(raw_data_path),
    output_dir=str(prepared_data_dir),
    tokenizer_name="meta-llama/Meta-Llama-3-8B-Instruct"
)

## 2. LoRA Training
Train a LoRA adapter. We use `train_lora.py` which leverages the `SFTTrainer` from the `trl` library along with `peft`.

In [ ]:
from src.arogya.models.finetune.train_lora import train_model

base_model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
adapter_output_dir = MODEL_DIR / "adapters" / "arogya-llama3-lora"

train_model(
    model_id=base_model_name,
    dataset_path=str(prepared_data_dir),
    output_dir=str(adapter_output_dir),
    num_train_epochs=3,
    per_device_train_batch_size=4,
    learning_rate=2e-4
)

## 3. Merging the Adapter
Merge the newly trained LoRA adapter back into the original base model for efficient inference.

In [ ]:
from src.arogya.models.finetune.merge_adapter import merge_adapters

merged_model_dir = MODEL_DIR / "merged" / "arogya-llama3-merged"

merge_adapters(
    base_model_id=base_model_name,
    adapter_path=str(adapter_output_dir),
    output_path=str(merged_model_dir)
)

## 4. Evaluation
Finally, evaluate the merged model using the metrics defined in `evaluate_model.py` to ensure it improved on our medical domain tasks.

In [ ]:
from src.arogya.models.finetune.evaluate_model import evaluate_model

test_data_path = DATA_DIR / "eval" / "medical_qa_test.jsonl"

results = evaluate_model(
    model_id=str(merged_model_dir),
    test_dataset_path=str(test_data_path)
)
print("Evaluation Results:", results)